# SE4050 Deep Learning (2026) — ResNet50 Transfer Learning & Fine-Tuning
### Member 2: Matheesha | Model: Deep Residual Network (ResNet50)

This notebook trains, fine-tunes, and evaluates the **ResNet50** architecture on the Food-101 benchmark under strict, fair experimental conditions.

#### Two-Stage Training Protocol:
1. **Phase 1 (Feature Extraction):** Frozen ImageNet backbone, training the custom classification head (`GlobalAveragePooling2D -> BatchNorm -> Dropout(0.3) -> Dense(101)`) with Adam ($lr=0.001$).
2. **Phase 2 (Fine-Tuning):** Unfreezing Stage 5 residual blocks (`conv5_block1_out` onward) with reduced learning rate ($lr=10^{-5}$) to adapt deep representations to fine-grained food textures.

In [ ]:
# Cell 1: Environment Setup & Automatic Git Sync
import os
import sys

REPO_NAME = "food-classification-deep-learning"
BRANCH_NAME = "feature/matheesha-resnet50"
REPO_URL = "https://github.com/niRmana11/food-classification-deep-learning.git"

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("[INFO] Running in Google Colab environment.")
    if not os.path.exists(f"/content/{REPO_NAME}"):
        print(f"[INFO] Cloning repository branch {BRANCH_NAME}...")
        !git clone -b {BRANCH_NAME} {REPO_URL}
        %cd /content/{REPO_NAME}
    else:
        %cd /content/{REPO_NAME}
        !git checkout {BRANCH_NAME}
        !git pull origin {BRANCH_NAME}
    
    if f"/content/{REPO_NAME}" not in sys.path:
        sys.path.insert(0, f"/content/{REPO_NAME}")
    
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print(f"[SUCCESS] GPU detected: {gpus[0].name}")
        !nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
    else:
        print("[WARNING] No GPU detected! Go to: Runtime -> Change runtime type -> T4 GPU")
else:
    print("[INFO] Running in local environment.")

In [ ]:
# Cell 2: Verify & Acquire Food-101 Dataset (~5GB archive)
!python -m src.data.download_food101

In [ ]:
# Cell 3: Load Shared Food-101 Data Pipeline (Model Type: resnet50)
import time
import tensorflow as tf
from src.preprocessing.data_loader import get_food101_datasets

BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

print("[INFO] Loading Food-101 datasets for ResNet50 (Caffe BGR zero-centering)...")
t0 = time.time()
train_ds, val_ds, test_ds = get_food101_datasets(
    data_dir="data/raw/food-101",
    splits_dir="data/splits",
    model_type="resnet50",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)
print(f"[SUCCESS] Datasets loaded in {time.time() - t0:.2f}s")

In [ ]:
# Cell 4: Instantiate ResNet50 Architecture (Phase 1: Feature Extraction)
from src.models.resnet50 import build_resnet50_model, compile_resnet50_model

model, base_model = build_resnet50_model(
    num_classes=101,
    input_shape=(224, 224, 3),
    dropout_rate=0.3
)

compile_resnet50_model(model, learning_rate=0.001)
model.summary()

In [ ]:
# Cell 5: Phase 1 Training (Feature Extraction — Frozen Backbone)
import os
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, CSVLogger

os.makedirs("results/resnet50", exist_ok=True)

callbacks_phase1 = [
    EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, min_lr=1e-5, verbose=1),
    CSVLogger("results/resnet50/history_phase1.csv")
]

PHASE1_EPOCHS = 6
print(f"[INFO] Starting Phase 1 training for {PHASE1_EPOCHS} epochs...")
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=PHASE1_EPOCHS,
    callbacks=callbacks_phase1
)

In [ ]:
# Cell 6: Phase 2 Fine-Tuning (Unfreezing Stage 5 Residual Blocks)
from src.models.resnet50 import unfreeze_resnet50_for_finetuning, compile_resnet50_model

unfrozen_count = unfreeze_resnet50_for_finetuning(model, base_model, fine_tune_from_layer="conv5_block1_out")
print(f"[INFO] Unfrozen {unfrozen_count} layers in ResNet50 backbone for fine-tuning.")

# Recompile with lower learning rate (1e-5) to protect pretrained representations
compile_resnet50_model(model, learning_rate=1e-5)
model.summary()

In [ ]:
# Cell 7: Phase 2 Training (Fine-Tuning Loop)
callbacks_phase2 = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-6, verbose=1),
    CSVLogger("results/resnet50/history_phase2.csv")
]

TOTAL_EPOCHS = 20
print(f"[INFO] Starting Phase 2 fine-tuning up to total epoch {TOTAL_EPOCHS}...")
history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=TOTAL_EPOCHS,
    initial_epoch=len(history_phase1.history["loss"]),
    callbacks=callbacks_phase2
)

In [ ]:
# Cell 8: Save Standardized 7 Evaluation Artifacts into results/resnet50/
print("[INFO] Model training complete. Proceed to standardized artifact generation.")
# Evaluated strictly on val_ds during development. Test set evaluated once final weights locked.